# Bearing fault diagnosis — 1D-CNN baseline (Colab)

Self-contained: no project files needed. **Use a GPU runtime** (Runtime → Change runtime type → T4 GPU); on a
CPU runtime the training takes many hours.

What it does, for the main operating condition (1500 rpm, 0.7 Nm, 1000 N):
1. Downloads the 29 bearings again (~4.6 GB, 20–40 min) and caches their 1-second vibration windows in Drive
   (`data/raw_windows_N15_M07_F10/`, ~0.6 GB). Skipped for bearings already cached.
2. Trains a WDCNN (Zhang et al., 2017) on raw vibration with the project's bearing-level splits:
   15 leave-one-bearing-out models (in-domain reference) and one model per calibration rotation (105).
3. As a sanity check, trains one model on a random split of windows (the same bearings in training and test).
4. Saves window probabilities to Drive (`data/cnn/`). Every model's output is saved as soon as it finishes, so if
   the session drops, run all cells again and it continues where it stopped.

At the end, download `cnn_results.zip` from Drive (`load_bearing_research_paper/data/`) and unzip it into the
project's `data/` folder, so that `data/cnn/lobo.csv` and `data/cnn/rotations/` exist.

Data licence: CC BY-NC 4.0 — cite Lessmeier et al. (2016) and the Paderborn KAt-DataCenter.

## 1. Setup

In [ ]:
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null

import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

DRIVE_DATA = Path("/content/drive/MyDrive/load_bearing_research_paper/data")
CACHE_DIR = DRIVE_DATA / "raw_windows_N15_M07_F10"   # cached 1 s windows, one .npz per bearing
OUT_DIR = DRIVE_DATA / "cnn"                          # CNN probabilities
FEATURES_CSV = DRIVE_DATA / "features" / "features_N15_M07_F10.csv"  # from colab_run.ipynb, used as a check

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "- switch to a GPU runtime!")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Bearings and splits (same as the project's `splits.py`)

In [ ]:
"""Bearings, bearing-level splits and calibration rotations for the CNN runs.

A self-contained copy of the project's split definitions (bearing_uq.bearings and bearing_uq.splits);
tests/test_cnn_protocol.py checks that both give the same splits and rotations.
"""

from itertools import product

import numpy as np
import pandas as pd

HEALTHY, INNER, OUTER = "healthy", "inner_race", "outer_race"
LABELS = [HEALTHY, INNER, OUTER]  # class index = position in this list

BEARINGS = pd.DataFrame(
    [(c, HEALTHY, "none") for c in ["K001", "K002", "K003", "K004", "K005", "K006"]]
    + [(c, OUTER, "artificial") for c in ["KA01", "KA03", "KA05", "KA06", "KA07", "KA08", "KA09"]]
    + [(c, INNER, "artificial") for c in ["KI01", "KI03", "KI05", "KI07", "KI08"]]
    + [(c, OUTER, "real") for c in ["KA04", "KA15", "KA16", "KA22", "KA30"]]
    + [(c, INNER, "real") for c in ["KI04", "KI14", "KI16", "KI17", "KI18", "KI21"]],
    columns=["bearing", "label", "origin"],
)

SOURCE_HEALTHY = ["K001", "K002", "K004"]
TARGET_HEALTHY = ["K003", "K005", "K006"]


def _codes(label, origin):
    rows = BEARINGS[(BEARINGS.label == label) & (BEARINGS.origin == origin)]
    return sorted(rows.bearing)


ARTIFICIAL_INNER = _codes(INNER, "artificial")
ARTIFICIAL_OUTER = _codes(OUTER, "artificial")
SOURCE = SOURCE_HEALTHY + ARTIFICIAL_OUTER + ARTIFICIAL_INNER
TARGET = TARGET_HEALTHY + _codes(OUTER, "real") + _codes(INNER, "real")


def rotations():
    """Yield (key, calibration bearings, training bearings) for all 3 x 5 x 7 = 105 rotations.

    The key ("K001+KI01+KA01") matches the file names used by the project's other experiments.
    """
    for calib in product(SOURCE_HEALTHY, ARTIFICIAL_INNER, ARTIFICIAL_OUTER):
        train = sorted(set(SOURCE) - set(calib))
        yield "+".join(calib), list(calib), train


def random_window_split(meta, test_fraction=0.2, seed=0):
    """Sanity check only: a random split of windows in which every bearing appears in both training and test.

    For each bearing (sorted by code), its windows (sorted by recording and window) are shuffled and the first
    test_fraction are marked as test. Returns a boolean array aligned with meta (True = test). Depends only on the
    window keys, so the project gives the same split for the feature table as Colab does for the raw windows.
    """
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(meta), dtype=bool)
    for code in sorted(meta["bearing"].unique()):
        rows = meta.index[meta["bearing"] == code]
        rows = meta.loc[rows].sort_values(["recording", "window"]).index.to_numpy()
        n_test = int(round(test_fraction * len(rows)))
        is_test[meta.index.get_indexer(rng.permutation(rows)[:n_test])] = True
    return is_test

## 4. Loading and caching windows

In [ ]:
"""Download Paderborn recordings and cache their 1-second vibration windows (one .npz per bearing).

Windows are cut exactly as in the feature table: recordings 1-20, non-overlapping 1 s windows from the
start of each recording, unreadable or missing recordings skipped.
"""

import shutil
import subprocess
import tempfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat

FS = 64_000
WINDOW = FS                 # 1-second windows
RECORDINGS = 20
CONDITION = "N15_M07_F10"   # main operating condition: 1500 rpm, 0.7 Nm, 1000 N
BASE_URL = "https://groups.uni-paderborn.de/kat/BearingDataCenter/"


def _channel_name(ch):
    name = ch["Name"]
    while isinstance(name, np.ndarray):
        name = name.flat[0]
    return str(name)


def load_vibration(path, channel="vibration_1"):
    record = loadmat(path)[Path(path).stem][0, 0]
    for ch in record["Y"].ravel():
        if _channel_name(ch) == channel:
            return np.asarray(ch["Data"], dtype=float).ravel()
    raise KeyError(f"{channel} not in {[_channel_name(c) for c in record['Y'].ravel()]}")


def windows_from_folder(folder, code, condition=CONDITION):
    """All 1 s windows of one bearing: (x float32 (n, WINDOW), recording numbers, window numbers)."""
    xs, recs, wins = [], [], []
    for n in range(1, RECORDINGS + 1):
        hits = list(Path(folder).rglob(f"{condition}_{code}_{n}.mat"))
        if not hits:
            print(f"  missing {condition}_{code}_{n}.mat")
            continue
        try:
            signal = load_vibration(hits[0])
        except Exception as e:  # a few Paderborn files are unreadable
            print(f"  skipped {hits[0].name}: {e}")
            continue
        for w in range(len(signal) // WINDOW):
            xs.append(signal[w * WINDOW:(w + 1) * WINDOW].astype(np.float32))
            recs.append(n)
            wins.append(w)
    return np.stack(xs), np.array(recs), np.array(wins)


def cache_bearing(code, cache_dir, condition=CONDITION):
    """Download one bearing's archive, extract it, and save its windows to <cache_dir>/<code>.npz."""
    with tempfile.TemporaryDirectory() as tmp:
        archive = Path(tmp) / f"{code}.rar"
        with urllib.request.urlopen(BASE_URL + f"{code}.rar") as r, open(archive, "wb") as f:
            shutil.copyfileobj(r, f, length=1 << 20)
        out = Path(tmp) / "x"
        out.mkdir()
        subprocess.run(["unrar", "x", "-o+", "-inul", str(archive), f"{out}/"], check=True)
        archive.unlink()
        x, rec, win = windows_from_folder(out, code, condition)
    np.savez(Path(cache_dir) / f"{code}.npz", x=x, recording=rec, window=win)
    return len(x)


def load_cache(cache_dir, bearings):
    """Stack the cached windows of all bearings: (x (N, WINDOW) float32, meta DataFrame)."""
    xs, metas = [], []
    for b in bearings.itertuples():
        d = np.load(Path(cache_dir) / f"{b.bearing}.npz")
        xs.append(d["x"])
        metas.append(pd.DataFrame({"bearing": b.bearing, "label": b.label, "origin": b.origin,
                                   "recording": d["recording"], "window": d["window"]}))
    return np.concatenate(xs), pd.concat(metas, ignore_index=True)

## 5. Model: WDCNN

In [ ]:
"""WDCNN: a 1D-CNN for raw vibration signals with a wide first-layer kernel.

Architecture of Zhang et al. (2017), "A new deep learning model for fault diagnosis with good anti-noise and
domain adaptation ability on raw vibration signals", Sensors 17(2), 425. The first convolution (kernel 64,
stride 16) acts as a learned filter bank; four small convolutions follow. Input: 4,096-sample segments.
"""

import torch
from torch import nn

SEGMENT = 4096


def _block(c_in, c_out, kernel, stride=1, padding=1):
    return [nn.Conv1d(c_in, c_out, kernel, stride, padding), nn.BatchNorm1d(c_out), nn.ReLU(),
            nn.MaxPool1d(2)]


class WDCNN(nn.Module):
    def __init__(self, n_classes=3, segment=SEGMENT):
        super().__init__()
        self.features = nn.Sequential(
            *_block(1, 16, 64, stride=16, padding=24),
            *_block(16, 32, 3), *_block(32, 64, 3), *_block(64, 64, 3),
            *_block(64, 64, 3, padding=0),
        )
        with torch.no_grad():
            n_flat = self.features(torch.zeros(1, 1, segment)).numel()
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(n_flat, 100), nn.BatchNorm1d(100), nn.ReLU(),
                                        nn.Linear(100, n_classes))

    def forward(self, x):  # x: (batch, segment)
        return self.classifier(self.features(x.unsqueeze(1)))

## 6. Training and prediction

In [ ]:
"""Train the CNN on random segments of 1 s windows; predict a window by averaging over its segments.

Settings are fixed in advance (no tuning on calibration or test bearings): 20 epochs, 8 random 4,096-sample
crops per training window per epoch, Adam (learning rate 1e-3, weight decay 1e-4), batch 256, class-balanced
cross-entropy. Each segment is standardised (zero mean, unit variance), so absolute vibration level is
removed. A window's probabilities are the mean softmax over its 15 non-overlapping segments.
"""

import numpy as np
import torch
import torch.nn.functional as F


EPOCHS = 20
CROPS_PER_WINDOW = 8
BATCH = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4


def standardise(seg):
    return (seg - seg.mean(dim=1, keepdim=True)) / (seg.std(dim=1, keepdim=True) + 1e-8)


def random_crops(x, idx, generator):
    """One random SEGMENT-long crop from each window x[idx]."""
    starts = torch.randint(0, x.shape[1] - SEGMENT + 1, (len(idx),), generator=generator, device=x.device)
    cols = starts[:, None] + torch.arange(SEGMENT, device=x.device)[None, :]
    return standardise(x[idx[:, None], cols])


def train_model(x, y, seed=0, epochs=EPOCHS, crops_per_window=CROPS_PER_WINDOW, batch=BATCH):
    """x: float tensor (N, window) on the training device; y: long tensor (N,) of class indices."""
    device = x.device
    torch.manual_seed(seed)
    gen = torch.Generator(device=device).manual_seed(seed)
    model = WDCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    counts = torch.bincount(y, minlength=3).float()
    weights = (counts.sum() / (len(counts) * counts.clamp(min=1))).to(device)

    model.train()
    for _ in range(epochs):
        order = torch.arange(len(y), device=device).repeat(crops_per_window)
        order = order[torch.randperm(len(order), generator=gen, device=device)]
        for start in range(0, len(order), batch):
            idx = order[start:start + batch]
            if len(idx) < 2:  # batch norm needs more than one sample
                continue
            loss = F.cross_entropy(model(random_crops(x, idx, gen)), y[idx], weight=weights)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model


@torch.no_grad()
def predict_windows(model, x, batch=64):
    """Mean softmax over the non-overlapping segments of each window: numpy (N, 3)."""
    model.eval()
    n_seg = x.shape[1] // SEGMENT
    out = []
    for start in range(0, len(x), batch):
        w = x[start:start + batch, :n_seg * SEGMENT].reshape(-1, SEGMENT)
        p = F.softmax(model(standardise(w)), dim=1).reshape(-1, n_seg, 3).mean(dim=1)
        out.append(p.double().cpu().numpy())
    p = np.concatenate(out)
    return p / p.sum(axis=1, keepdims=True)

## 7. Experiment runner

In [ ]:
"""Run the CNN experiments and save window probabilities as CSV files (resumable).

lobo.csv              - each source bearing predicted by a CNN trained on the other 14 source bearings
rotations/<key>.csv   - for each of the 105 rotations, a CNN trained on that rotation's training bearings
                        predicts every bearing it was not trained on (calibration and all target bearings)
random_split.csv      - sanity check: a CNN trained on a random 80 % of the source bearings' windows predicts the
                        other 20 % (the same bearings appear in training and test)
Columns: bearing, recording, window, p_healthy, p_inner_race, p_outer_race.
"""

import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch


PROB_COLS = [f"p_{label}" for label in LABELS]


class Data:
    """All cached windows on the training device, with their bearing, recording and window numbers."""

    def __init__(self, x, meta, device):
        self.meta = meta.reset_index(drop=True)
        self.x = torch.as_tensor(x, device=device)
        self.y = torch.tensor(meta["label"].map({l: i for i, l in enumerate(LABELS)}).to_numpy(),
                              device=device)

    def index(self, codes):
        return torch.as_tensor(np.flatnonzero(self.meta["bearing"].isin(codes).to_numpy()),
                               device=self.x.device)


def _fit(data, codes, seed):
    idx = data.index(codes)
    return train_model(data.x[idx], data.y[idx], seed=seed)


def _predict(model, data, codes):
    idx = data.index(codes)
    rows = data.meta.iloc[idx.cpu().numpy()][["bearing", "recording", "window"]].reset_index(drop=True)
    return pd.concat([rows, pd.DataFrame(predict_windows(model, data.x[idx]), columns=PROB_COLS)], axis=1)


def _save(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, float_format="%.10g")
    tmp.replace(path)  # never leave a half-written file behind


def run_lobo(data, out_dir, seed=0):
    """Leave one source bearing out (in-domain reference). One file per held-out bearing, then lobo.csv."""
    folder = Path(out_dir) / "lobo"
    folder.mkdir(parents=True, exist_ok=True)
    for i, code in enumerate(SOURCE, start=1):
        path = folder / f"{code}.csv"
        if path.exists():
            continue
        t0 = time.time()
        model = _fit(data, [c for c in SOURCE if c != code], seed)
        _save(_predict(model, data, [code]), path)
        print(f"lobo {i}/{len(SOURCE)} {code} ({time.time() - t0:.0f} s)", flush=True)
    frames = [pd.read_csv(folder / f"{c}.csv") for c in SOURCE]
    _save(pd.concat(frames, ignore_index=True), Path(out_dir) / "lobo.csv")


def run_rotations(data, out_dir, seed=0):
    folder = Path(out_dir) / "rotations"
    folder.mkdir(parents=True, exist_ok=True)
    all_codes = list(data.meta["bearing"].unique())
    rots = list(rotations())
    for i, (key, calib, train) in enumerate(rots, start=1):
        path = folder / f"{key}.csv"
        if path.exists():
            continue
        t0 = time.time()
        model = _fit(data, train, seed)
        _save(_predict(model, data, [c for c in all_codes if c not in train]), path)
        print(f"rotation {i}/{len(rots)} {key} ({time.time() - t0:.0f} s)", flush=True)


def run_random_split(data, out_dir, seed=0):
    """Sanity check with a random window split of the source bearings (not a bearing-level split)."""
    path = Path(out_dir) / "random_split.csv"
    if path.exists():
        print("random split: already done")
        return
    source = data.meta[data.meta["bearing"].isin(SOURCE)]
    is_test = random_window_split(source, seed=seed)
    train_idx = torch.as_tensor(source.index[~is_test].to_numpy(), device=data.x.device)
    test_rows = source.index[is_test].to_numpy()
    t0 = time.time()
    model = train_model(data.x[train_idx], data.y[train_idx], seed=seed)
    probs = predict_windows(model, data.x[torch.as_tensor(test_rows, device=data.x.device)])
    keys = data.meta.loc[test_rows, ["bearing", "recording", "window"]].reset_index(drop=True)
    _save(pd.concat([keys, pd.DataFrame(probs, columns=PROB_COLS)], axis=1), path)
    acc = (probs.argmax(1) == data.y[torch.as_tensor(test_rows, device=data.x.device)].cpu().numpy()).mean()
    print(f"random split: test accuracy {acc:.3f} on {len(test_rows)} windows ({time.time() - t0:.0f} s)")

## 8. Download and cache the windows (20–40 min, once)

In [ ]:
for b in BEARINGS.itertuples():
    if (CACHE_DIR / f"{b.bearing}.npz").exists():
        print(f"{b.bearing}: already cached")
        continue
    try:
        print(f"{b.bearing}: {cache_bearing(b.bearing, CACHE_DIR)} windows")
    except Exception as e:
        print(f"{b.bearing}: FAILED - {e}  (run this cell again)")

## 9. Load the windows onto the GPU

In [ ]:
x, meta = load_cache(CACHE_DIR, BEARINGS)
print(f"{len(meta)} windows from {meta.bearing.nunique()} bearings")

# The windows must be the same as in the feature table used by the other models.
if FEATURES_CSV.exists():
    keys = ["bearing", "recording", "window"]
    feat = pd.read_csv(FEATURES_CSV, usecols=keys)
    same = feat.merge(meta[keys], how="outer", indicator=True)["_merge"].value_counts()
    print("windows only in features:", same.get("left_only", 0), "| only here:", same.get("right_only", 0))
else:
    print("feature table not found in Drive; skipping the check")

data = Data(x, meta, DEVICE)
del x

## 10. In-domain reference: leave one source bearing out (15 models)

In [ ]:
run_lobo(data, OUT_DIR)

## 11. Artificial → real: one model per calibration rotation (105 models)

In [ ]:
run_rotations(data, OUT_DIR)

## 12. Sanity check: random window split (1 model)

Trains on a random 80 % of the source bearings' windows and tests on the other 20 %, so the same bearings appear in training and test. High accuracy here shows the network works; the bearing-level results show how much of that is bearing identity.

In [ ]:
run_random_split(data, OUT_DIR)

## 13. Package the results

In [ ]:
done = len(list((OUT_DIR / "rotations").glob("*.csv")))
print(f"{done}/105 rotations, lobo.csv present: {(OUT_DIR / 'lobo.csv').exists()}")
if done == 105:
    shutil.make_archive(str(DRIVE_DATA / "cnn_results"), "zip", root_dir=DRIVE_DATA, base_dir="cnn")
    print("wrote", DRIVE_DATA / "cnn_results.zip")